# Task 1 — Direct Clarity Classification

Fine‑tuning di **Llama 3.1 8B‑Instruct** (4‑bit QLoRA + DoRA) per classificare direttamente la chiarezza di una risposta politica nelle 3 macro‑categorie: **Ambivalent**, **Clear Reply**, **Clear Non‑Reply**.

In [ ]:
# ============================================================
# CELLA 1 — Setup Ambiente
# ============================================================

# Installazione dipendenze (decommentare su Colab)
# !pip install -q -U transformers peft trl bitsandbytes datasets accelerate

import os, sys, json, gc
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from huggingface_hub import login
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer
from datasets import load_from_disk
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

# Mount Google Drive + login HF (solo su Colab)
try:
    import google.colab
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=True)

    REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"
    hf_cache_dir = "/content/drive/MyDrive/progettoLLM/hf_cache"
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Login HF tramite Colab Secrets
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print(f"✅ Colab configurato. Cache: {hf_cache_dir}")

except ImportError:
    # Ambiente locale: legge il token da .env
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    os.environ['HF_TOKEN'] = hf_token
                    login(token=hf_token)
                    break
    print("✅ Ambiente locale rilevato.")

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ============================================================
# CELLA 2 — Variabili e Prompt
# ============================================================

# ── Flag di controllo ─────────────────────────────────────────
ESEGUI_TRAINING = False   # True = addestra da zero, False = carica checkpoint

# ── Modello e percorsi ────────────────────────────────────────
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
PATH_MODELLO_SALVATO = "./risultati_clarity_task1/modello_finale"
OUTPUT_DIR = "./risultati_clarity_task1"

# ── Precisione: bfloat16 se supportato (L4/A100), altrimenti float16 (T4) ──
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {COMPUTE_DTYPE}")

# ── System prompt per la classificazione diretta a 3 classi ──
SYSTEM_PROMPT_TASK1 = """Sei un esperto analista di discorsi politici. Il tuo compito è classificare la risposta di un politico a una domanda diretta in una delle seguenti 3 categorie:

1. 'Clear Reply': La risposta ammette una sola interpretazione chiara.
2. 'Ambivalent': Viene fornita una risposta, ma è strutturata in modo da prestarsi a molteplici interpretazioni o essere vaga.
3. 'Clear Non-Reply': Il rispondente rifiuta apertamente o dichiara di non poter fornire le informazioni richieste.

Rispondi SOLO con il nome esatto della categoria (Clear Reply, Ambivalent o Clear Non-Reply), senza aggiungere altro testo."""

# Etichette nell'ordine usato per i report
LABELS = ["Ambivalent", "Clear Reply", "Clear Non-Reply"]

print("✅ Configurazione caricata.")

In [ ]:
# ============================================================
# CELLA 3 — Dataset Management
# ============================================================

# Caricamento tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Caricamento split train e test
train_dataset = load_from_disk("dataset/QEvasion/train")
test_dataset  = load_from_disk("dataset/QEvasion/test")
print(f"Train: {len(train_dataset)} esempi | Test: {len(test_dataset)} esempi")


def format_prompt_task1(example):
    """Formatta un esempio nel template chat di Llama 3.1 per il Task 1."""
    domanda  = example.get('interview_question', '')
    risposta = example.get('interview_answer', '')
    label    = example.get('clarity_label', '')

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT_TASK1},
        {"role": "user",      "content": f"Domanda: {domanda}\nRisposta del politico: {risposta}"},
        {"role": "assistant", "content": str(label)},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


# Formattazione del training set
formatted_train = train_dataset.map(format_prompt_task1, remove_columns=train_dataset.column_names)

print("\n--- Esempio di prompt formattato ---")
print(formatted_train[0]["text"][:500] + "\n...")

In [ ]:
# ============================================================
# CELLA 4 — Core Logico (Training o Loading)
# ============================================================

# Pulizia preventiva VRAM per evitare OOM
torch.cuda.empty_cache()
gc.collect()

# Configurazione quantizzazione 4-bit comune
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

if ESEGUI_TRAINING:
    # ── TRAINING ──────────────────────────────────────────────
    print("⚙️  Modalità TRAINING attiva.")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto",
        quantization_config=bnb_config, torch_dtype=COMPUTE_DTYPE,
    )
    model = prepare_model_for_kbit_training(model)

    # Configurazione DoRA sui layer di attenzione di Llama
    peft_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", use_dora=True,
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        optim="paged_adamw_32bit",
        save_strategy="epoch",
        logging_steps=10,
        learning_rate=2e-4,
        weight_decay=0.001,
        fp16=(COMPUTE_DTYPE == torch.float16),
        bf16=(COMPUTE_DTYPE == torch.bfloat16),
        max_grad_norm=0.3,
        num_train_epochs=3,
        warmup_steps=100,
        group_by_length=True,
        lr_scheduler_type="cosine",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_train,
        args=training_args,
        peft_config=None,          # modello già wrappato con PEFT
        processing_class=tokenizer,
    )

    print("🚀 Inizio addestramento...")
    trainer.train()

    # Salvataggio adattatori + tokenizer su Drive
    trainer.model.save_pretrained(PATH_MODELLO_SALVATO)
    tokenizer.save_pretrained(PATH_MODELLO_SALVATO)
    print(f"✅ Modello salvato in: {PATH_MODELLO_SALVATO}")

else:
    # ── CARICAMENTO DA CHECKPOINT ─────────────────────────────
    print(f"📦 Caricamento modello base ({MODEL_ID}) in 4-bit...")

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto",
        quantization_config=bnb_config, torch_dtype=COMPUTE_DTYPE,
    )

    print(f"🔗 Applicazione adattatori LoRA da: {PATH_MODELLO_SALVATO}")
    model = PeftModel.from_pretrained(
        base_model, PATH_MODELLO_SALVATO,
        low_cpu_mem_usage=True, device_map="auto",
    )

    # Sovrascriviamo il tokenizer con quello salvato col checkpoint
    tokenizer = AutoTokenizer.from_pretrained(PATH_MODELLO_SALVATO)
    print("✅ Modello e adattatori caricati.")

model.eval()
print(f"Modello in eval mode. Device: {model.device}")

In [ ]:
# ============================================================
# CELLA 5 — Funzione di Inferenza
# ============================================================

def predici_clarity(domanda: str, risposta: str) -> str:
    """
    Predice la macro-categoria di chiarezza per una coppia domanda/risposta.
    Restituisce una stringa tra: 'Clear Reply', 'Ambivalent', 'Clear Non-Reply'.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_TASK1},
        {"role": "user",   "content": f"Domanda: {domanda}\nRisposta del politico: {risposta}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


# Sanity check rapido
print("--- Sanity check (3 esempi) ---")
for i in range(min(3, len(test_dataset))):
    es = test_dataset[i]
    pred = predici_clarity(es['interview_question'], es['interview_answer'])
    vero = str(es.get('clarity_label', '')).strip()
    ok = '✅' if pred == vero else '❌'
    print(f"  [{i+1}] Pred: {pred:20s} | Vera: {vero:20s} {ok}")

In [ ]:
# ============================================================
# CELLA 6 — Analisi Performance
# ============================================================

y_true, y_pred = [], []

print(f"Valutazione su {len(test_dataset)} esempi...")

for i in tqdm(range(len(test_dataset))):
    esempio = test_dataset[i]
    y_true.append(str(esempio.get('clarity_label', '')).strip())

    try:
        pred = predici_clarity(esempio['interview_question'], esempio['interview_answer'])
        y_pred.append(pred)
    except Exception as e:
        print(f"⚠️ Errore esempio {i}: {e}")
        y_pred.append("Error")

# ── Classification Report (3 cifre decimali) ─────────────────
print("\n" + "=" * 60)
print("REPORT FINALE — TASK 1 (CLARITY CLASSIFICATION)")
print("=" * 60)
print(classification_report(y_true, y_pred, labels=LABELS, digits=3))

# ── Matrice di Confusione ─────────────────────────────────────
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred, labels=LABELS)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('Predizioni LLM')
plt.ylabel('Etichette Reali')
plt.title('Matrice di Confusione — Task 1 (Direct Clarity)')
plt.tight_layout()
plt.show()